In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [3]:
# --------------------------------------------------
# 1. Dataset
# --------------------------------------------------

data = [
    ("i am a student", "je suis etudiant"),
    ("i like cats", "j aime les chats"),
    ("i love dogs", "j aime les chiens"),
    ("i am happy", "je suis heureux"),
    ("i like music", "j aime la musique"),
]

In [7]:
# --------------------------------------------------
# 2. Build vocabulary
# --------------------------------------------------

def build_vocab(sentences):

    vocab = {
        "<pad>": 0,
        "<sos>": 1,
        "<end>": 2,
        "<unk>": 3
    }

    for sentence in sentences:

        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

src_vocab = build_vocab([x[0] for x in data])
trg_vocab = build_vocab(x[1] for x in data)

print(src_vocab)
print(trg_vocab)

{'<pad>': 0, '<sos>': 1, '<end>': 2, '<unk>': 3, 'i': 4, 'am': 5, 'a': 6, 'student': 7, 'like': 8, 'cats': 9, 'love': 10, 'dogs': 11, 'happy': 12, 'music': 13}
{'<pad>': 0, '<sos>': 1, '<end>': 2, '<unk>': 3, 'je': 4, 'suis': 5, 'etudiant': 6, 'j': 7, 'aime': 8, 'les': 9, 'chats': 10, 'chiens': 11, 'heureux': 12, 'la': 13, 'musique': 14}


In [9]:
def numericalize(sentence, vocab):
    token = ["<sos>"] + sentence.split() + ["<eos>"]
    return torch.tensor(
        [vocab.get(word, vocab["<unk>"]) for word in token],
        dtype=torch.long
    )

src_seq = [numericalize(sentence, src_vocab) for sentence, _ in data]
trg_seq = [numericalize(sentence, trg_vocab) for _, sentence in data]

print(src_seq)
print(trg_seq)

[tensor([1, 4, 5, 6, 7, 3]), tensor([1, 4, 8, 9, 3]), tensor([ 1,  4, 10, 11,  3]), tensor([ 1,  4,  5, 12,  3]), tensor([ 1,  4,  8, 13,  3])]
[tensor([1, 4, 5, 6, 3]), tensor([ 1,  7,  8,  9, 10,  3]), tensor([ 1,  7,  8,  9, 11,  3]), tensor([ 1,  4,  5, 12,  3]), tensor([ 1,  7,  8, 13, 14,  3])]


In [12]:
from torch.nn.utils.rnn import pad_sequence

src = pad_sequence(
    src_seq, batch_first=True, padding_value=src_vocab['<pad>']
)
trg = pad_sequence(
    trg_seq, batch_first=True, padding_value=trg_vocab['<pad>']
)

print(src.shape, trg.shape)
print(src, trg)

torch.Size([5, 6]) torch.Size([5, 6])
tensor([[ 1,  4,  5,  6,  7,  3],
        [ 1,  4,  8,  9,  3,  0],
        [ 1,  4, 10, 11,  3,  0],
        [ 1,  4,  5, 12,  3,  0],
        [ 1,  4,  8, 13,  3,  0]]) tensor([[ 1,  4,  5,  6,  3,  0],
        [ 1,  7,  8,  9, 10,  3],
        [ 1,  7,  8,  9, 11,  3],
        [ 1,  4,  5, 12,  3,  0],
        [ 1,  7,  8, 13, 14,  3]])


In [14]:
class Encoder(nn.Module):

    def __init__(self, input_dim, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim,
            embedding_dim
        )

        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, x):
        embedding = self.embedding(x)
        return self.gru(embedding)

In [18]:
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim, embedding_dim):
        super().__init__()

        self.outpu_dim = output_dim
        self.embedding = nn.Embedding(
            output_dim,
            embedding_dim
        )

        self.gru = nn.GRU(
            embedding_dim + hidden_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            embedding_dim + hidden_dim * 2,
            output_dim
        )

    def forward(self, input, hidden):
        input = input.unsqueeze(1)

        embedded = self.embedding(input)
        output, hidden = self.rnn(embedded, hidden)

        output = output.squeeze(1)

        prediction = self.fc(output)

        return prediction, hidden

In [19]:
class Seq2Seq(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):

        batch_size = src.size(0)

        trg_len = trg.size(1)

        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            batch_size,
            trg_len,
            trg_vocab_size
        ).to(src.device)

        encoder_outputs, hidden = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):

            output, hidden = self.decoder(
                input,
                hidden,
                encoder_outputs
            )

            outputs[:, t] = output

            teacher_force = (random.random() < teacher_forcing_ratio)

            top1 = output.argmax(1)

            input = (
                trg[:, t] if teacher_force else top1)

        return outputs